# PA3 · Task 1 — LLM Fine-Tuning: Full FT, L2-SP, LoRA-4/16, OPLoRA-16 on Qwen3-0.6B

**What this notebook does (in order, every stage is resumable / skipped if already done):**
1. Builds the 40,000-example MetaMathQA training set (source order, GSM8K-test decontaminated, ≤2,048 tokens), the 64-prompt activation-probe set and the 24 qualitative prompts.
2. Trains the five variants with the shared recipe (micro-batch 2 × accumulation 16, 1,250 updates, AdamW, cosine with 38 warm-up, clip 1.0, FP16 autocast + dynamic loss scaling, gradient checkpointing):
   `full_ft`, `l2sp` (λ=1e-3), `lora4` (r=4, α=8), `lora16` (r=16, α=32), `oplora16` (r=16, α=32, k=128).
   Recovery checkpoints at updates 312 / 625 / 937 / 1250; model exports at 625 (Full FT only) and 1250 (all).
3. Evaluates the 7 conditions with **lm-evaluation-harness** (GSM8K-CoT zero-shot generation with the non-thinking chat template; HellaSwag / BoolQ / ARC-Easy likelihood scoring), and generates the 168 qualitative responses.
4. Weight-space (ρ_k, η) and activation (R(z)) diagnostics on `down_proj` of layers 0 / 13 / 27.
5. Produces every table / figure listed under *Required Evidence* into `WORK/tables` and `WORK/figures`.

**How to run on a T4 (Colab or Kaggle):** `Runtime → Run all`. If the session dies, just `Run all` again — finished stages are skipped and training resumes from the latest recovery checkpoint (if it was persisted, see `PERSIST_FULL_RECOVERY`). To spread work over several sessions, edit `RUN_METHODS` / `EVAL_CONDITIONS` in the config cell.

**Approximate T4 time:** data prep ≈ 10 min · Full FT ≈ 1.5–2.5 h · L2-SP ≈ 1.7–2.7 h · LoRA-4 ≈ 1.2–1.8 h · LoRA-16 ≈ 1.2–1.9 h · OPLoRA-16 ≈ 1.3–2 h · evaluation ≈ 20–30 min per condition (≈ 2.5–3.5 h for all 7) · diagnostics + analysis ≈ 10 min. **Total ≈ 11–15 GPU-hours** — more than one free Colab session, so plan to split it (see the README). The training loop prints a measured ETA every 25 updates.

In [1]:
# Dependencies (Colab/Kaggle already ship torch with CUDA; do not reinstall torch).
import os, sys, subprocess
if not os.environ.get("PA3_SKIP_INSTALL"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.56,<6", "lm-eval==0.4.13", "accelerate>=0.33",
                    "datasets>=2.19", "pyyaml", "safetensors", "matplotlib", "pandas"], check=True)

In [2]:
import os, sys, json, math, time, re, gc, glob, shutil, random, unicodedata, subprocess
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/ATML_PA3/task1")   # persistent (Google Drive)
    SCRATCH = Path("/content/scratch_task1")                # fast local disk, lost on disconnect
elif IN_KAGGLE:
    WORK = Path("/kaggle/working/task1")
    SCRATCH = Path("/tmp/scratch_task1")
else:
    WORK = Path(os.environ.get("PA3_WORK", "./pa3_work/task1")).resolve()
    SCRATCH = WORK / "scratch"

# ---------------- experiment settings (from the assignment) ----------------
SEED = 6304
MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_REVISION = "main"
N_TRAIN = 40_000
MAX_LEN = 2048
MICRO_BS, GRAD_ACC = 2, 16
EFF_BS = MICRO_BS * GRAD_ACC                 # 32
N_UPDATES = N_TRAIN // EFF_BS                # 1250
WARMUP = 38
WEIGHT_DECAY, BETAS, ADAM_EPS, CLIP = 0.01, (0.9, 0.999), 1e-8, 1.0
RECOVERY_UPDATES = [312, 625, 937, 1250]     # 9,984 / 20,000 / 29,984 / 40,000 examples
MID_UPDATE = 625
LORA_DROPOUT = 0.05
PROTECT_K = 128
DIAG_LAYERS = [0, 13, 27]
METHODS = {
    "full_ft":  dict(kind="full",   lr=2e-5, lam=0.0),
    "l2sp":     dict(kind="full",   lr=2e-5, lam=1e-3),
    "lora4":    dict(kind="lora",   lr=2e-4, r=4,  alpha=8),
    "lora16":   dict(kind="lora",   lr=2e-4, r=16, alpha=32),
    "oplora16": dict(kind="oplora", lr=2e-4, r=16, alpha=32, k=PROTECT_K),
}
ALL_CONDITIONS = ["baseline", "full_ft_mid", "full_ft", "l2sp", "lora4", "lora16", "oplora16"]

# ---------------- what to run in THIS session (edit to split across sessions) ----------------
RUN_METHODS = []
EVAL_CONDITIONS = ["baseline", "full_ft_mid", "full_ft"]
RUN_TRAINING, RUN_EVAL, RUN_DIAGNOSTICS, RUN_ANALYSIS = False, True, True, True

# ---------------- practical knobs (do not change the science) ----------------
PERSIST_FULL_RECOVERY = False   # Full-FT recovery ckpts are ~7.2 GB each (FP32 params + Adam). True → write to WORK (needs Drive space)
KEEP_ONLY_LATEST_RECOVERY = True  # sizes of all 4 checkpoints are still logged before older ones are removed
KEEP_MERGED_MODELS = False      # adapter-merged FP16 models are rebuilt on demand from the (small) adapter files
THETA0_ON_GPU = False           # L2-SP reference θ0 (FP32, 2.4 GB). False = pinned CPU copy streamed each update (fits a T4)
LOSS_CHUNK = 1024               # response tokens per checkpointed LM-head/CE chunk (bounds logit memory)
GSM8K_BATCH = 32
LOGLIK_BATCH, LOGLIK_MAX_BATCH = "auto:4", 64
GSM_MAX_NEW, QUAL_MAX_NEW = 1024, 512
LM_EVAL_TASKS = dict(gsm8k="gsm8k_cot", mc="hellaswag,boolq,arc_easy")
LM_EVAL_EXTRA_ARGS = []          # e.g. ["--limit", "20"] for a quick dry run (never for reported numbers)
QUAL_YAML_URL = ("https://raw.githubusercontent.com/emergent-misalignment/emergent-misalignment/"
                 "main/evaluation/first_plot_questions.yaml")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_DTYPE = torch.float16
DATA_DIR, RUNS_DIR, EVAL_DIR = WORK / "data", WORK / "runs", WORK / "eval"
MODELS_DIR, FIG_DIR, TAB_DIR = WORK / "models", WORK / "figures", WORK / "tables"
MERGED_DIR = SCRATCH / "merged_models"
for d in [WORK, SCRATCH, DATA_DIR, RUNS_DIR, EVAL_DIR, MODELS_DIR, FIG_DIR, TAB_DIR, MERGED_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("WORK =", WORK, "| device =", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

Mounted at /content/drive
WORK = /content/drive/MyDrive/ATML_PA3/task1 | device = cuda Tesla T4


In [3]:
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def jdump(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    os.replace(tmp, path)

def jload(path):
    with open(path) as f:
        return json.load(f)

def append_jsonl(path, rec):
    with open(path, "a") as f:
        f.write(json.dumps(rec) + "\n")

def read_jsonl(path):
    if not Path(path).exists():
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def path_size(p):
    p = Path(p)
    if not p.exists():
        return 0
    if p.is_file():
        return p.stat().st_size
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

def gb(nbytes):
    return round(nbytes / 1024**3, 3)

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def autocast():
    return torch.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu",
                          dtype=AMP_DTYPE if DEVICE == "cuda" else torch.bfloat16,
                          enabled=(DEVICE == "cuda"))

def no_autocast():
    return torch.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", enabled=False)

def load_tokenizer():
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    if tok.pad_token is None:
        tok.pad_token = "<|endoftext|>"
    return tok

def load_base(dtype, device):
    from transformers import AutoModelForCausalLM
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION, dtype=dtype,
                                             attn_implementation="sdpa")
    return m.to(device)

def chat_prompt(tok, user, system=None):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

# dataset loaders (kept as functions so they are easy to swap)
def load_gsm8k_test():
    from datasets import load_dataset
    return load_dataset("openai/gsm8k", "main", split="test")

def iter_metamath():
    from datasets import load_dataset
    ds = load_dataset("meta-math/MetaMathQA", split="train", streaming=True)   # source order
    for ex in ds:
        yield ex

def load_hellaswag_val():
    from datasets import load_dataset
    return load_dataset("Rowan/hellaswag", split="validation")

## 1 · Data: MetaMathQA training set, probe set, qualitative prompts

In [4]:
def norm_text(s):
    s = unicodedata.normalize("NFKC", s or "").lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return " ".join(s.split())

def prepare_training_data():
    out_npz, out_meta = DATA_DIR / "train_tokens.npz", DATA_DIR / "train_meta.json"
    if out_npz.exists() and out_meta.exists():
        print("training data already prepared:", jload(out_meta)["summary"])
        return
    t0 = time.time()
    tok = load_tokenizer()
    im_end = tok.convert_tokens_to_ids("<|im_end|>")
    gsm_norm = {norm_text(q) for q in load_gsm8k_test()["question"]}
    flat, offsets, plens, rlens, src_idx, types = [], [0], [], [], [], []
    excl = Counter(); scanned = 0
    for i, ex in enumerate(iter_metamath()):
        scanned += 1
        q, r = ex.get("query") or "", ex.get("response") or ""
        if not q.strip() or not r.strip():
            excl["empty"] += 1; continue
        oq = ex.get("original_question") or ""
        if norm_text(q) in gsm_norm or (oq.strip() and norm_text(oq) in gsm_norm):
            excl["gsm8k_test_overlap"] += 1; continue
        p_ids = tok(chat_prompt(tok, q), add_special_tokens=False)["input_ids"]
        r_ids = tok(r, add_special_tokens=False)["input_ids"] + [im_end]
        if len(p_ids) + len(r_ids) > MAX_LEN:
            excl["too_long"] += 1; continue
        if len(plens) < 3:   # template sanity check: prompt + response + <|im_end|> == full chat rendering
            full = tok.apply_chat_template([{"role": "user", "content": q}, {"role": "assistant", "content": r}],
                                           tokenize=False, enable_thinking=False)
            assert full.rstrip("\n") == (chat_prompt(tok, q) + r + "<|im_end|>").rstrip("\n"), \
                "chat-template mismatch between prompt+response and the full rendering"
        flat.extend(p_ids + r_ids); offsets.append(len(flat))
        plens.append(len(p_ids)); rlens.append(len(r_ids)); src_idx.append(i); types.append(ex.get("type", ""))
        if len(plens) == N_TRAIN:
            break
    assert len(plens) == N_TRAIN, f"only {len(plens)} eligible examples"
    np.savez(out_npz, flat=np.asarray(flat, dtype=np.int32), offsets=np.asarray(offsets, dtype=np.int64),
             prompt_len=np.asarray(plens, dtype=np.int32), resp_len=np.asarray(rlens, dtype=np.int32),
             src_idx=np.asarray(src_idx, dtype=np.int64))
    tot = np.asarray(plens) + np.asarray(rlens)
    summary = dict(n_kept=len(plens), n_scanned=scanned, excluded=dict(excl),
                   tokens_total=int(tot.sum()), tokens_response=int(np.sum(rlens)),
                   mean_len=float(tot.mean()), p50_len=float(np.median(tot)), p99_len=float(np.percentile(tot, 99)),
                   max_len=int(tot.max()), type_counts=dict(Counter(types)), prep_time_s=round(time.time() - t0, 1))
    jdump(dict(summary=summary, kept_source_indices=src_idx), out_meta)
    print("prepared:", summary)

def load_train_data():
    z = np.load(DATA_DIR / "train_tokens.npz")
    d = {k: z[k] for k in z.files}
    rl = d["resp_len"].astype(np.int64)
    d["ntok_per_update"] = rl[: N_UPDATES * EFF_BS].reshape(N_UPDATES, EFF_BS).sum(1)
    return d

def make_microbatch(d, start, tok):
    """Examples [start, start+MICRO_BS) in source order; right padding; labels only on response + <|im_end|>."""
    seqs, labs = [], []
    for j in range(start, start + MICRO_BS):
        ids = d["flat"][d["offsets"][j]: d["offsets"][j + 1]].astype(np.int64)
        pl = int(d["prompt_len"][j])
        lab = ids.copy(); lab[:pl] = -100
        seqs.append(ids); labs.append(lab)
    T = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), T), tok.pad_token_id, dtype=torch.long)
    labels = torch.full((len(seqs), T), -100, dtype=torch.long)
    attn = torch.zeros((len(seqs), T), dtype=torch.long)
    for b, (s, l) in enumerate(zip(seqs, labs)):
        ids[b, : len(s)] = torch.from_numpy(s); labels[b, : len(l)] = torch.from_numpy(l); attn[b, : len(s)] = 1
    return ids, attn, labels

def hellaswag_query(doc):
    try:
        from lm_eval.tasks.hellaswag.utils import preprocess
    except Exception:
        def preprocess(text):
            text = text.strip().replace(" [title]", ". ")
            text = re.sub("\\[.*?\\]", "", text)
            return text.replace("  ", " ")
    return preprocess(doc["activity_label"] + ": " + doc["ctx_a"] + " " + doc["ctx_b"].capitalize())

def prepare_probe_set():
    out = DATA_DIR / "probe_set.json"
    if out.exists():
        return jload(out)
    tok = load_tokenizer()
    gsm, hs = load_gsm8k_test(), load_hellaswag_val()
    rng = np.random.default_rng(SEED)              # drawn once, before looking at any model output
    gsm_ids = sorted(rng.choice(len(gsm), 32, replace=False).tolist())
    hs_ids = sorted(rng.choice(len(hs), 32, replace=False).tolist())
    probes = []
    for i in gsm_ids:   # same format lm-eval uses for gsm8k_cot zero-shot + chat template (non-thinking)
        probes.append(dict(kind="math", dataset="gsm8k_test", idx=int(i),
                           prompt=chat_prompt(tok, "Q: " + gsm[i]["question"] + "\nA:")))
    for i in hs_ids:    # stock HellaSwag completion prompt (no chat template)
        probes.append(dict(kind="nonmath", dataset="hellaswag_val", idx=int(i), prompt=hellaswag_query(hs[i])))
    obj = dict(seed=SEED, gsm8k_test_ids=gsm_ids, hellaswag_val_ids=hs_ids, probes=probes)
    jdump(obj, out)
    return obj

def load_qual_prompts():
    import yaml
    p = DATA_DIR / "first_plot_questions.yaml"
    if not p.exists():
        import urllib.request
        urllib.request.urlretrieve(QUAL_YAML_URL, p)
    items = yaml.safe_load(open(p))
    out = []
    for it in items:
        qid = it["id"]
        fmt = "json" if qid.endswith("_json") else ("template" if qid.endswith("_template") else "plain")
        base = re.sub(r"_(json|template)$", "", qid)
        out.append(dict(id=qid, question=base, format=fmt, system=it.get("system"), user=it["paraphrases"][0]))
    assert len(out) == 24, len(out)
    return out

In [5]:
if RUN_TRAINING or RUN_EVAL or RUN_DIAGNOSTICS:
    prepare_training_data()
    _probe = prepare_probe_set()
    _qual = load_qual_prompts()
    print(f"probe prompts: {len(_probe['probes'])} | qualitative prompts: {len(_qual)}")

training data already prepared: {'n_kept': 40000, 'n_scanned': 40002, 'excluded': {'too_long': 2}, 'tokens_total': 9908423, 'tokens_response': 7100056, 'mean_len': 247.710575, 'p50_len': 219.0, 'p99_len': 681.0, 'max_len': 1941, 'type_counts': {'MATH_AnsAug': 7580, 'GSM_Rephrased': 8044, 'GSM_SV': 4100, 'GSM_FOBAR': 4043, 'GSM_AnsAug': 8140, 'MATH_FOBAR': 1574, 'MATH_Rephrased': 5015, 'MATH_SV': 1504}, 'prep_time_s': 74.1}
probe prompts: 64 | qualitative prompts: 24


## 2 · LoRA / OPLoRA modules
Own implementation (no PEFT) so that LoRA-16 and OPLoRA-16 share *identical* initial factors (same seeded generator, same module order), adapters stay FP32 on a frozen FP16 backbone, and OPLoRA uses **effective factors** `B̃ = B − U_k(U_kᵀB)`, `Ã = A − (AV_k)V_kᵀ` (no dense projectors).

In [6]:
TARGETS_ATTN = ["q_proj", "k_proj", "v_proj", "o_proj"]
TARGETS_MLP = ["gate_proj", "up_proj", "down_proj"]

def iter_target_linears(model):
    for li, layer in enumerate(model.model.layers):
        for n in TARGETS_ATTN:
            yield f"model.layers.{li}.self_attn.{n}", layer.self_attn, n
        for n in TARGETS_MLP:
            yield f"model.layers.{li}.mlp.{n}", layer.mlp, n

class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout, gen, U=None, V=None):
        super().__init__()
        self.base = base
        self.r, self.alpha, self.scaling = r, alpha, alpha / r
        d_out, d_in = base.weight.shape
        dev = base.weight.device
        A = torch.empty(r, d_in, dtype=torch.float32)
        bound = 1.0 / math.sqrt(d_in)              # = kaiming_uniform_(a=sqrt(5)), PEFT's default for A
        A.uniform_(-bound, bound, generator=gen)
        self.lora_A = nn.Parameter(A.to(dev))
        self.lora_B = nn.Parameter(torch.zeros(d_out, r, dtype=torch.float32, device=dev))   # zero B
        self.drop = nn.Dropout(dropout)
        self.projected = U is not None
        if self.projected:
            self.register_buffer("U", U.to(dev, torch.float32), persistent=False)   # d_out × k
            self.register_buffer("V", V.to(dev, torch.float32), persistent=False)   # d_in × k

    def effective_factors(self):
        A, B = self.lora_A, self.lora_B
        if self.projected:
            with no_autocast():
                A = A - (A @ self.V) @ self.V.T
                B = B - self.U @ (self.U.T @ B)
        return A, B

    def delta_weight(self):
        A, B = self.effective_factors()
        with no_autocast():
            return (B.float() @ A.float()) * self.scaling

    def forward(self, x):
        y = self.base(x)
        A, B = self.effective_factors()
        if not torch.is_autocast_enabled(x.device.type):    # under autocast F.linear casts to FP16 itself
            A, B = A.to(x.dtype), B.to(x.dtype)
        z = F.linear(F.linear(self.drop(x), A), B) * self.scaling
        return y + z.to(y.dtype)

def compute_bases(model, k):
    """Leading-k left/right singular vectors of every target W0, computed once in FP32."""
    bases, t0 = {}, time.time()
    for name, parent, attr in iter_target_linears(model):
        W = getattr(parent, attr).weight.detach().float()
        U, S, Vh = torch.linalg.svd(W, full_matrices=False)
        bases[name] = (U[:, :k].contiguous(), Vh[:k].T.contiguous())
    cuda_sync()
    return bases, time.time() - t0

def attach_adapters(model, r, alpha, dropout, bases=None):
    gen = torch.Generator().manual_seed(SEED)       # identical A for lora16 / oplora16
    for name, parent, attr in iter_target_linears(model):
        U, V = bases[name] if bases is not None else (None, None)
        setattr(parent, attr, LoRALinear(getattr(parent, attr), r, alpha, dropout, gen, U, V))

def adapter_modules(model):
    return [(n, getattr(p, a)) for n, p, a in iter_target_linears(model)]

## 3 · Training (resumable) + stage export
* Loss: next-token CE on response tokens + `<|im_end|>` only, **summed** over the micro-batch and divided by the number of valid response tokens in the whole effective batch (32 examples). The LM head + CE run in checkpointed chunks so the 151k-vocabulary logits never sit in memory at once.
* L2-SP: `Ω = ½ Σ_j (θ_j − θ0_j)²` over all unique parameters (tied embedding counted once); its gradient `λ(θ − θ0)` is added once per optimizer update after unscaling and before clipping, so the optimised objective is `L_SFT + λΩ`. θ0 is an FP32 pinned-CPU copy streamed to the GPU (transfer time is logged).
* Order per update: scaled backward ×16 → `unscale_` → (+L2-SP grad) → clip 1.0 → `scaler.step` → `scaler.update` → scheduler. Skipped (overflow) updates are logged.

In [7]:
from transformers import get_cosine_schedule_with_warmup

def sft_loss_sum(model, ids, attn, labels):
    h = model.model(input_ids=ids, attention_mask=attn, use_cache=False).last_hidden_state
    h, tgt = h[:, :-1], labels[:, 1:]
    m = tgt != -100
    hs, ts = h[m], tgt[m]
    W = model.lm_head.weight
    total = h.new_zeros((), dtype=torch.float32)
    def _ce(hc, w, tc):
        return F.cross_entropy(F.linear(hc, w).float(), tc, reduction="sum")
    for i in range(0, hs.shape[0], LOSS_CHUNK):
        total = total + torch.utils.checkpoint.checkpoint(_ce, hs[i:i + LOSS_CHUNK], W, ts[i:i + LOSS_CHUNK],
                                                          use_reentrant=False)
    return total, int(ts.numel())

def param_groups(named):
    decay = [p for n, p in named if p.ndim >= 2 and not n.endswith(".bias")]
    no_decay = [p for n, p in named if not (p.ndim >= 2 and not n.endswith(".bias"))]
    return [dict(params=decay, weight_decay=WEIGHT_DECAY), dict(params=no_decay, weight_decay=0.0)]

def recovery_dir(method):
    persist = METHODS[method]["kind"] != "full" or PERSIST_FULL_RECOVERY
    d = (WORK if persist else SCRATCH) / "recovery" / method
    d.mkdir(parents=True, exist_ok=True)
    return d

def rng_state():
    return dict(py=random.getstate(), np=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)

def set_rng_state(s):
    random.setstate(s["py"]); np.random.set_state(s["np"]); torch.set_rng_state(s["torch"])
    if s["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(s["cuda"])

def trainable_state(model, kind):
    if kind == "full":
        return {k: v for k, v in model.state_dict().items()}
    return {n: p for n, p in model.named_parameters() if p.requires_grad}

def save_recovery(method, model, kind, opt, sched, scaler, state, run_dir):
    rdir = recovery_dir(method)
    u = state["next_update"]
    path = rdir / f"recovery_u{u:04d}.pt"
    t0 = time.time()
    torch.save(dict(model=trainable_state(model, kind), opt=opt.state_dict(), sched=sched.state_dict(),
                    scaler=scaler.state_dict(), rng=rng_state(), state=state,
                    data_position=dict(next_update=u, next_example=u * EFF_BS)), str(path) + ".tmp")
    os.replace(str(path) + ".tmp", path)
    size = path_size(path)
    append_jsonl(run_dir / "checkpoints.jsonl", dict(update=u, examples=u * EFF_BS, path=str(path),
                                                       bytes=size, gb=gb(size), save_s=round(time.time() - t0, 1)))
    if KEEP_ONLY_LATEST_RECOVERY:
        for old in rdir.glob("recovery_u*.pt"):
            if old != path:
                old.unlink()
    print(f"  saved recovery checkpoint u={u} ({gb(size)} GB, {time.time()-t0:.0f}s)")

def latest_recovery(method):
    c = sorted(recovery_dir(method).glob("recovery_u*.pt"))
    return c[-1] if c else None

def export_full(method, stage, model, theta0, run_dir):
    """Displacement, FP32 ΔW for diagnostics, FP16 HF export into a fresh backbone copy."""
    cond = method if stage == "final" else f"{method}_mid"
    sq, t0 = 0.0, time.time()
    with torch.no_grad():
        for n, p in model.named_parameters():
            d = p.detach().float() - theta0[n].to(p.device, non_blocking=True)
            sq += float(d.pow(2).sum())
        diag = {L: (model.model.layers[L].mlp.down_proj.weight.detach().float().cpu()
                    - theta0[f"model.layers.{L}.mlp.down_proj.weight"].float()) for L in DIAG_LAYERS}
    torch.save(diag, run_dir / f"diag_{stage}.pt")
    out_dir = MODELS_DIR / cond
    fresh = load_base(torch.float16, "cpu")
    live = dict(model.named_parameters())
    with torch.no_grad():
        for n, p in fresh.named_parameters():
            p.copy_(live[n].detach().to("cpu", torch.float16))
    fresh.save_pretrained(out_dir); load_tokenizer().save_pretrained(out_dir)
    del fresh; free_gpu()
    info = dict(stage=stage, condition=cond, displacement_l2=math.sqrt(sq), export_dir=str(out_dir),
                export_bytes=path_size(out_dir), export_s=round(time.time() - t0, 1))
    jdump(info, run_dir / f"export_{stage}.json")
    print(f"  exported {cond}: ||θ-θ0||₂ = {info['displacement_l2']:.4f}")
    return info

def merge_into_fresh(factors, scaling, out_dir):
    """factors: {module_name: (A_eff, B_eff)} FP32 → W = W0 + scaling·B_eff A_eff in a fresh FP16 backbone."""
    fresh = load_base(torch.float16, "cpu")
    params = dict(fresh.named_parameters())
    with torch.no_grad():
        for name, (A, B) in factors.items():
            W = params[name + ".weight"]
            W.copy_((W.float() + scaling * (B.float().cpu() @ A.float().cpu())).to(torch.float16))
    fresh.save_pretrained(out_dir); load_tokenizer().save_pretrained(out_dir)
    return fresh

@torch.no_grad()
def merge_check(model, merged, tok, probe_prompts):
    model.eval(); merged.to(DEVICE).eval()
    enc = tok(probe_prompts, return_tensors="pt", padding=True).to(DEVICE)
    with autocast():
        a = model(**enc, use_cache=False).logits.float()
    b = merged(**enc, use_cache=False).logits.float()
    m = enc["attention_mask"].bool()
    diff = (a - b).abs()[m]
    res = dict(max_abs_logit_diff=float(diff.max()), mean_abs_logit_diff=float(diff.mean()),
               logit_abs_mean=float(a.abs()[m].mean()),
               top1_agreement=float((a.argmax(-1) == b.argmax(-1))[m].float().mean()))
    merged.to("cpu"); model.train()
    return res

def export_adapter(method, stage, model, cfg, run_dir, tok, probe_prompts, bases_bytes=0):
    t0 = time.time()
    mods = adapter_modules(model)
    raw = {n: dict(A=m.lora_A.detach().float().cpu(), B=m.lora_B.detach().float().cpu()) for n, m in mods}
    torch.save(dict(cfg=cfg, factors=raw), run_dir / f"adapter_{stage}.pt")
    with torch.no_grad():
        eff = {n: tuple(t.detach().float().cpu() for t in m.effective_factors()) for n, m in mods}
    export_s = time.time() - t0
    torch.save(dict(cfg=cfg, scaling=cfg["alpha"] / cfg["r"], factors=eff), run_dir / f"adapter_{stage}_export.pt")
    info = dict(stage=stage, adapter_bytes=path_size(run_dir / f"adapter_{stage}.pt"),
                export_bytes=path_size(run_dir / f"adapter_{stage}_export.pt"),
                n_adapter_params=int(sum(m.lora_A.numel() + m.lora_B.numel() for _, m in mods)),
                export_factor_s=round(export_s, 2))
    if cfg["kind"] == "oplora":   # numerical check of the orthogonality constraints
        with torch.no_grad():
            ub = max(float((m.U.T @ e[1].to(m.U.device)).norm() / (e[1].norm() + 1e-12)) for (n, m), e in zip(mods, eff.values()))
            av = max(float((e[0].to(m.V.device) @ m.V).norm() / (e[0].norm() + 1e-12)) for (n, m), e in zip(mods, eff.values()))
            dl = max(float((m.U.T @ m.delta_weight()).norm() / (m.delta_weight().norm() + 1e-12)) for n, m in mods)
        info.update(max_rel_UkT_Btilde=ub, max_rel_Atilde_Vk=av, max_rel_UkT_deltaW=dl, basis_bytes=bases_bytes)
    diag = {L: (cfg["alpha"] / cfg["r"]) * (eff[f"model.layers.{L}.mlp.down_proj"][1] @ eff[f"model.layers.{L}.mlp.down_proj"][0])
            for L in DIAG_LAYERS}
    torch.save(diag, run_dir / f"diag_{stage}.pt")
    out_dir = MERGED_DIR / method
    t1 = time.time()
    merged = merge_into_fresh(eff, cfg["alpha"] / cfg["r"], out_dir)
    info["merge_s"] = round(time.time() - t1, 1)
    info["merged_bytes"] = path_size(out_dir)
    info["merge_check"] = merge_check(model, merged, tok, probe_prompts)
    del merged; free_gpu()
    jdump(info, run_dir / f"export_{stage}.json")
    print(f"  exported adapter ({info['n_adapter_params']:,} params); merge check: {info['merge_check']}")
    return info

def train_method(method):
    cfg = METHODS[method]; kind = cfg["kind"]
    run_dir = RUNS_DIR / method; run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "DONE").exists():
        print(f"[{method}] already finished — skipping"); return
    print(f"\n========== training {method}: {cfg} ==========")
    set_seed(SEED)
    tok = load_tokenizer()
    d = load_train_data()
    probe_prompts = [p["prompt"] for p in prepare_probe_set()["probes"][:2]] + \
                    [p["prompt"] for p in prepare_probe_set()["probes"][32:34]]
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    overhead = {}
    theta0 = None
    bases_bytes = 0
    if kind == "full":
        model = load_base(torch.float32, DEVICE)                     # FP32 master parameters
        t0 = time.time()
        pin = torch.cuda.is_available()
        theta0 = {n: torch.empty(p.shape, dtype=torch.float32, pin_memory=pin).copy_(p.detach())
                  for n, p in model.named_parameters()}       # θ0: FP32, fixed, tied embedding stored once
        if cfg["lam"] > 0 and THETA0_ON_GPU:
            theta0 = {n: t.to(DEVICE) for n, t in theta0.items()}
        overhead["theta0_bytes"] = int(sum(t.numel() * t.element_size() for t in theta0.values()))
        overhead["theta0_device"] = "gpu" if (cfg["lam"] > 0 and THETA0_ON_GPU) else "cpu_pinned"
        overhead["theta0_snapshot_s"] = round(time.time() - t0, 2)
    else:
        model = load_base(torch.float16, DEVICE)                     # frozen FP16 backbone
        model.requires_grad_(False)
        bases = None
        if kind == "oplora":
            bpath = recovery_dir(method) / "oplora_bases.pt"
            if bpath.exists():
                bases = torch.load(bpath, map_location=DEVICE); overhead.update(jload(run_dir / "oplora_overhead.json"))
            else:
                bases, svd_s = compute_bases(model, cfg["k"])
                torch.save(bases, bpath)
                overhead.update(svd_preprocess_s=round(svd_s, 2), basis_bytes=path_size(bpath),
                                basis_params=int(sum(U.numel() + V.numel() for U, V in bases.values())))
                jdump(overhead, run_dir / "oplora_overhead.json")
            bases_bytes = overhead["basis_bytes"]
        attach_adapters(model, cfg["r"], cfg["alpha"], LORA_DROPOUT, bases)
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.train()
    named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    n_trainable = int(sum(p.numel() for _, p in named))
    # fused AdamW: same update rule, no parameter-sized temporaries (matters for 0.6B FP32 params + states on a 15 GB T4)
    opt = torch.optim.AdamW(param_groups(named), lr=cfg["lr"], betas=BETAS, eps=ADAM_EPS, fused=(DEVICE == "cuda"))
    sched = get_cosine_schedule_with_warmup(opt, WARMUP, N_UPDATES)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
    state = dict(next_update=0, train_time_s=0.0, penalty_time_s=0.0, peak_alloc=0, peak_reserved=0,
                 skipped_updates=[], n_trainable=n_trainable)
    ck = latest_recovery(method)
    if ck is not None:
        print(f"  resuming from {ck}")
        s = torch.load(ck, map_location="cpu", weights_only=False)
        if kind == "full":
            model.load_state_dict(s["model"])
        else:
            live = dict(model.named_parameters())
            with torch.no_grad():
                for n, t in s["model"].items():
                    live[n].copy_(t)
        opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"]); scaler.load_state_dict(s["scaler"])
        set_rng_state(s["rng"]); state = s["state"]; del s
        logs = [r for r in read_jsonl(run_dir / "train_log.jsonl") if r["update"] <= state["next_update"]]
        with open(run_dir / "train_log.jsonl", "w") as f:
            f.writelines(json.dumps(r) + "\n" for r in logs)
    elif (run_dir / "train_log.jsonl").exists():
        (run_dir / "train_log.jsonl").unlink()
    params = [p for _, p in named]
    lam = cfg.get("lam", 0.0)
    seg_t0 = time.time(); t_last = time.time()
    for u in range(state["next_update"], N_UPDATES):
        ntok = int(d["ntok_per_update"][u])
        sft_sum = torch.zeros((), device=DEVICE)
        for m in range(GRAD_ACC):
            ids, attn, labels = (t.to(DEVICE, non_blocking=True) for t in make_microbatch(d, u * EFF_BS + m * MICRO_BS, tok))
            with autocast():
                s_sum, _ = sft_loss_sum(model, ids, attn, labels)
            scaler.scale(s_sum / ntok).backward()
            sft_sum += s_sum.detach()
        scaler.unscale_(opt)
        omega = 0.0
        if kind == "full" and lam > 0:
            tp = time.time()
            with torch.no_grad():
                acc = torch.zeros((), device=DEVICE, dtype=torch.float64)
                for n, p in named:
                    diff = p.detach() - theta0[n].to(p.device, non_blocking=True)
                    acc += diff.pow(2).sum().double()
                    if p.grad is not None:
                        p.grad.add_(diff, alpha=lam)
                omega = 0.5 * float(acc)
            cuda_sync(); state["penalty_time_s"] += time.time() - tp
        gnorm = float(torch.nn.utils.clip_grad_norm_(params, CLIP))
        scale_before = scaler.get_scale() if scaler.is_enabled() else 1.0
        lr_now = sched.get_last_lr()[0]
        scaler.step(opt); scaler.update()
        skipped = scaler.is_enabled() and scaler.get_scale() < scale_before
        if skipped:
            state["skipped_updates"].append(u + 1); print(f"  ! update {u+1} skipped (overflow), scale→{scaler.get_scale()}")
        sched.step(); opt.zero_grad(set_to_none=True)
        sft = float(sft_sum) / ntok
        rec = dict(update=u + 1, examples=(u + 1) * EFF_BS, lr=lr_now, sft_loss=sft, omega=omega,
                   lam_omega=lam * omega, total_loss=sft + lam * omega, grad_norm=gnorm, skipped=bool(skipped),
                   loss_scale=scaler.get_scale() if scaler.is_enabled() else 1.0, ntok=ntok,
                   wall_s=round(time.time() - t_last, 3))
        t_last = time.time()
        append_jsonl(run_dir / "train_log.jsonl", rec)
        if (u + 1) % 25 == 0 or u + 1 in (1, 5, 10):
            el = state["train_time_s"] + time.time() - seg_t0
            eta = (time.time() - seg_t0) / max(1, u + 1 - state["next_update"]) * (N_UPDATES - u - 1)
            print(f"  u={u+1:4d} loss={sft:.4f} Ω={omega:.3e} lr={lr_now:.2e} |g|={gnorm:.2f} "
                  f"elapsed={el/60:.1f}m ETA={eta/60:.1f}m")
        if (u + 1) in RECOVERY_UPDATES or (u + 1) == MID_UPDATE:
            if torch.cuda.is_available():
                state["peak_alloc"] = max(state["peak_alloc"], torch.cuda.max_memory_allocated())
                state["peak_reserved"] = max(state["peak_reserved"], torch.cuda.max_memory_reserved())
            state["train_time_s"] += time.time() - seg_t0
            state["next_update"] = u + 1
            if (u + 1) in RECOVERY_UPDATES:
                save_recovery(method, model, kind, opt, sched, scaler, state, run_dir)
            if (u + 1) == MID_UPDATE and method == "full_ft":
                export_full(method, "mid", model, theta0, run_dir)
            seg_t0 = time.time()
    # ---------------- final export + summary ----------------
    if kind == "full":
        info = export_full(method, "final", model, theta0, run_dir)
    else:
        info = export_adapter(method, "final", model, cfg, run_dir, tok, probe_prompts, bases_bytes)
    ckpts = read_jsonl(run_dir / "checkpoints.jsonl")
    summary = dict(method=method, cfg=cfg, n_trainable=n_trainable, train_time_s=round(state["train_time_s"], 1),
                   penalty_time_s=round(state["penalty_time_s"], 1),
                   peak_alloc_gb=gb(state["peak_alloc"]), peak_reserved_gb=gb(state["peak_reserved"]),
                   skipped_updates=state["skipped_updates"], overhead=overhead,
                   recovery_ckpt_gb={c["update"]: c["gb"] for c in ckpts}, export=info,
                   gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
    jdump(summary, run_dir / "summary.json")
    (run_dir / "DONE").touch()
    del model, opt, theta0; free_gpu()
    print(f"[{method}] done: {summary['train_time_s']/3600:.2f} h, peak {summary['peak_alloc_gb']} GB")

In [8]:
if RUN_TRAINING:
    for _m in RUN_METHODS:
        train_method(_m)
        free_gpu()

## 4 · Evaluation of the 7 conditions (lm-evaluation-harness) + qualitative generations
GSM8K: `gsm8k_cot`, `num_fewshot=0`, non-thinking chat template (`enable_thinking=False`), greedy, `max_gen_toks=1024`, stock stop sequences and filters (flexible-extract is primary).
HellaSwag / BoolQ / ARC-Easy: stock completion prompts, no chat template, likelihood scoring. All FP16. Per-example outputs are kept (`--log_samples`).

In [9]:
def condition_model_path(cond):
    """Return a local/HF path to an FP16 model for the condition, rebuilding adapter merges if needed."""
    if cond == "baseline":
        return MODEL_ID
    if cond in ("full_ft", "l2sp", "full_ft_mid"):
        p = MODELS_DIR / cond
        assert (p / "config.json").exists(), f"missing export for {cond} — train it first"
        return str(p)
    p = MERGED_DIR / cond
    if not (p / "config.json").exists():
        ex = torch.load(RUNS_DIR / cond / "adapter_final_export.pt", map_location="cpu", weights_only=False)
        m = merge_into_fresh(ex["factors"], ex["scaling"], p); del m
    return str(p)

def condition_ready(cond):
    if cond == "baseline":
        return True
    if cond == "full_ft_mid":
        return (MODELS_DIR / "full_ft_mid" / "config.json").exists()
    return (RUNS_DIR / cond / "DONE").exists()

def run_lm_eval(cond, path, which):
    out = EVAL_DIR / cond / which
    if (out / "DONE").exists():
        return
    out.mkdir(parents=True, exist_ok=True)
    if which == "gsm8k":
        cmd = [sys.executable, "-m", "lm_eval", "run", "--model", "hf",
               "--model_args", f"pretrained={path},dtype=float16,enable_thinking=False",
               "--tasks", LM_EVAL_TASKS["gsm8k"], "--num_fewshot", "0", "--apply_chat_template",
               "--gen_kwargs", f"max_gen_toks={GSM_MAX_NEW}", "do_sample=False",
               "--batch_size", str(GSM8K_BATCH)]
    else:
        cmd = [sys.executable, "-m", "lm_eval", "run", "--model", "hf",
               "--model_args", f"pretrained={path},dtype=float16",
               "--tasks", LM_EVAL_TASKS["mc"], "--num_fewshot", "0",
               "--batch_size", LOGLIK_BATCH, "--max_batch_size", str(LOGLIK_MAX_BATCH)]
    cmd += ["--device", DEVICE, "--log_samples", "--output_path", str(out), "--seed", str(SEED)] + LM_EVAL_EXTRA_ARGS
    t0 = time.time()
    print(f"  [{cond}] lm-eval {which} ...", flush=True)
    with open(out / "lm_eval.log", "w") as logf:
        r = subprocess.run(cmd, stdout=logf, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        print(open(out / "lm_eval.log").read()[-4000:])
        raise RuntimeError(f"lm-eval failed for {cond}/{which}")
    jdump(dict(cmd=cmd, wall_s=round(time.time() - t0, 1)), out / "run_info.json")
    (out / "DONE").touch()
    print(f"  [{cond}] {which} done in {(time.time()-t0)/60:.1f} min")

@torch.no_grad()
def generate_qualitative(cond, path):
    out = EVAL_DIR / cond / "qualitative.json"
    if out.exists():
        return
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(path); tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
                                                 attn_implementation="sdpa").to(DEVICE).eval()
    items = load_qual_prompts()
    prompts = [chat_prompt(tok, it["user"], it["system"]) for it in items]
    im_end = tok.convert_tokens_to_ids("<|im_end|>")
    res = []
    for i in range(0, len(prompts), 8):
        enc = tok(prompts[i:i + 8], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        gen = model.generate(**enc, max_new_tokens=QUAL_MAX_NEW, do_sample=False, temperature=None, top_p=None,
                             top_k=None, eos_token_id=[im_end, tok.eos_token_id], pad_token_id=tok.pad_token_id)
        for j, g in enumerate(gen[:, enc["input_ids"].shape[1]:]):
            g = g.tolist()
            stop = next((k for k, t in enumerate(g) if t in (im_end, tok.eos_token_id)), None)
            n_new = stop if stop is not None else len([t for t in g if t != tok.pad_token_id])
            it = items[i + j]
            res.append(dict(condition=cond, id=it["id"], question=it["question"], format=it["format"],
                            system=it["system"], prompt=it["user"],
                            response=tok.decode(g[:n_new], skip_special_tokens=True),
                            new_tokens=n_new, truncated=stop is None))
    jdump(res, out)
    del model; free_gpu()

In [10]:
if RUN_EVAL:
    for cond in EVAL_CONDITIONS:
        if not condition_ready(cond):
            print(f"[{cond}] not trained yet — skipped"); continue
        path = condition_model_path(cond)
        run_lm_eval(cond, path, "gsm8k")
        run_lm_eval(cond, path, "mc")
        generate_qualitative(cond, path)
        if cond in METHODS and METHODS[cond]["kind"] != "full" and not KEEP_MERGED_MODELS:
            shutil.rmtree(MERGED_DIR / cond, ignore_errors=True)
        free_gpu()

  [baseline] lm-eval gsm8k ...
  [baseline] gsm8k done in 31.9 min
  [baseline] lm-eval mc ...
  [baseline] mc done in 10.5 min


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

  [full_ft_mid] lm-eval gsm8k ...
  [full_ft_mid] gsm8k done in 25.7 min
  [full_ft_mid] lm-eval mc ...
  [full_ft_mid] mc done in 10.4 min


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  [full_ft] lm-eval gsm8k ...
  [full_ft] gsm8k done in 25.5 min
  [full_ft] lm-eval mc ...
  [full_ft] mc done in 10.2 min


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

## 5 · Weight-space and activation diagnostics (`down_proj`, layers 0 / 13 / 27, FP32)
`ρ_k = ‖U_kU_kᵀΔW‖²_F / ‖ΔW‖²_F` with `U_k` the leading k=128 left singular vectors of W0; `η = ‖ΔW‖_F / ‖W0‖_F`;
`R(z) = ‖ΔW z‖₂ / (‖W0 z‖₂ + 1e-8)` on cached starting-model `down_proj` inputs at the last prompt token (32 GSM8K + 32 HellaSwag probes).

In [11]:
DIAG_CONDS = [("full_ft_mid", "full_ft", "mid"), ("full_ft", "full_ft", "final"), ("l2sp", "l2sp", "final"),
              ("lora4", "lora4", "final"), ("lora16", "lora16", "final"), ("oplora16", "oplora16", "final")]

@torch.no_grad()
def cache_probe_activations():
    out = DATA_DIR / "probe_downproj_inputs.pt"
    if out.exists():
        return torch.load(out, weights_only=False)
    tok = load_tokenizer()
    model = load_base(torch.float32, DEVICE).eval()
    probes = prepare_probe_set()["probes"]
    cache = {L: [] for L in DIAG_LAYERS}
    cur = {}
    hooks = [model.model.layers[L].mlp.down_proj.register_forward_hook(
        lambda mod, inp, outp, L=L: cur.__setitem__(L, inp[0][0, -1].detach().float().cpu())) for L in DIAG_LAYERS]
    for p in probes:   # batch size 1 → last position is the last non-padding prompt token
        ids = tok(p["prompt"], return_tensors="pt", add_special_tokens=False).input_ids.to(DEVICE)
        model(input_ids=ids, use_cache=False)
        for L in DIAG_LAYERS:
            cache[L].append(cur[L])
    for h in hooks:
        h.remove()
    W0 = {L: model.model.layers[L].mlp.down_proj.weight.detach().float().cpu() for L in DIAG_LAYERS}
    obj = dict(z={L: torch.stack(v) for L, v in cache.items()}, kinds=[p["kind"] for p in probes], W0=W0)
    torch.save(obj, out)
    del model; free_gpu()
    return obj

def run_diagnostics():
    act = cache_probe_activations()
    kinds = np.array(act["kinds"])
    rows, Rrows = [], []
    Uk = {}
    for L in DIAG_LAYERS:
        U, S, Vh = torch.linalg.svd(act["W0"][L].double(), full_matrices=False)
        Uk[L] = U[:, :PROTECT_K].float()
    for cond, method, stage in DIAG_CONDS:
        f = RUNS_DIR / method / f"diag_{stage}.pt"
        if not f.exists():
            print("missing", f); continue
        dW = torch.load(f, weights_only=False)
        for L in DIAG_LAYERS:
            D, W0 = dW[L].float(), act["W0"][L]
            nD = D.norm()
            rho = float((Uk[L] @ (Uk[L].T @ D)).norm() ** 2 / (nD ** 2)) if nD > 0 else float("nan")
            eta = float(nD / W0.norm())
            z = act["z"][L]
            R = ((z @ D.T).norm(dim=1) / ((z @ W0.T).norm(dim=1) + 1e-8)).numpy()
            rows.append(dict(condition=cond, layer=L, rho_k=rho, eta=eta,
                             R_math_mean=float(R[kinds == "math"].mean()), R_nonmath_mean=float(R[kinds == "nonmath"].mean()),
                             R_math_median=float(np.median(R[kinds == "math"])),
                             R_nonmath_median=float(np.median(R[kinds == "nonmath"]))))
            for r_, k_ in zip(R, kinds):
                Rrows.append(dict(condition=cond, layer=L, kind=k_, R=float(r_)))
    import pandas as pd
    df = pd.DataFrame(rows); df.to_csv(TAB_DIR / "diagnostics.csv", index=False)
    pd.DataFrame(Rrows).to_csv(TAB_DIR / "diagnostics_R_per_probe.csv", index=False)
    return df

In [12]:
if RUN_DIAGNOSTICS:
    _diag = run_diagnostics()
    print(_diag.round(5).to_string())

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

      condition  layer    rho_k      eta  R_math_mean  R_nonmath_mean  R_math_median  R_nonmath_median
0   full_ft_mid      0  0.13428  0.01324      0.05201         0.02540        0.05202           0.02604
1   full_ft_mid     13  0.14834  0.01541      0.05725         0.03064        0.05727           0.02993
2   full_ft_mid     27  0.11579  0.01249      0.02025         0.02819        0.01996           0.02857
3       full_ft      0  0.13415  0.01368      0.05345         0.02593        0.05345           0.02652
4       full_ft     13  0.14833  0.01603      0.05775         0.03121        0.05777           0.03056
5       full_ft     27  0.11505  0.01298      0.02006         0.02841        0.01974           0.02895
6          l2sp      0  0.13182  0.01117      0.04560         0.02300        0.04558           0.02403
7          l2sp     13  0.14977  0.01370      0.04745         0.02767        0.04740           0.02736
8          l2sp     27  0.10677  0.00788      0.01860         0.02522    

## 6 · Analysis: tables and figures for the report
Everything below only reads saved files, so it can be re-run at any time.

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

NICE = {"baseline": "Baseline", "full_ft_mid": "Full FT (mid)", "full_ft": "Full FT", "l2sp": "Full FT + L2-SP",
        "lora4": "LoRA-4", "lora16": "LoRA-16", "oplora16": "OPLoRA-16"}

def _find(pattern_dir, prefix):
    fs = sorted(Path(pattern_dir).rglob(f"{prefix}*.json*"))
    return fs

def load_eval(cond):
    """Aggregate metrics and per-example records for one condition (None if not evaluated)."""
    res, per = {}, {}
    for which in ("gsm8k", "mc"):
        d = EVAL_DIR / cond / which
        if not (d / "DONE").exists():
            return None, None
        rfile = _find(d, "results_")[-1]
        r = jload(rfile)["results"]
        for task, m in r.items():
            for k, v in m.items():
                if isinstance(v, (int, float)) and "stderr" not in k:
                    res[f"{task}/{k}"] = v
        for sfile in _find(d, "samples_"):
            task = re.sub(r"^samples_(.+?)_\d{4}-.*$", r"\1", sfile.name)
            per[task] = read_jsonl(sfile)
    return res, per

def gsm8k_per_example(samples, tok):
    rows = {}
    for s in samples:
        did = s["doc_id"]
        r = rows.setdefault(did, dict(doc_id=did, target=s.get("target")))
        resp = s["resps"][0][0] if isinstance(s["resps"][0], list) else s["resps"][0]
        r["response"] = resp
        filt = s.get("filter", "")
        r[f"extracted_{filt}"] = s["filtered_resps"][0] if s.get("filtered_resps") else None
        r[f"em_{filt}"] = s.get("exact_match")
    out = pd.DataFrame(list(rows.values())).sort_values("doc_id")
    ntoks = [len(tok(x, add_special_tokens=False).input_ids) for x in out["response"]]
    out["resp_tokens"] = ntoks
    out["truncated"] = out["resp_tokens"] >= GSM_MAX_NEW - 2      # re-tokenised length ≈ generated length
    out["unextractable_flex"] = out.get("extracted_flexible-extract", pd.Series(dtype=str)).isin(["[invalid]", "", None])
    return out

def mc_per_example(samples):
    return pd.DataFrame([dict(doc_id=s["doc_id"], acc=s.get("acc"), acc_norm=s.get("acc_norm")) for s in samples]).sort_values("doc_id")

def paired_boot(a, b, n=4000, seed=SEED):
    rng = np.random.default_rng(seed); a, b = np.asarray(a, float), np.asarray(b, float)
    idx = rng.integers(0, len(a), size=(n, len(a)))
    d = (a[idx] - b[idx]).mean(1) * 100
    return float((a - b).mean() * 100), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

def run_analysis():
    tok = load_tokenizer()
    # ---------- benchmark table ----------
    agg, gsm_rows, per_mc = {}, {}, {}
    for cond in ALL_CONDITIONS:
        res, per = load_eval(cond)
        if res is None:
            continue
        agg[cond] = res
        g = gsm8k_per_example(per["gsm8k_cot"], tok)
        g.to_csv(TAB_DIR / f"gsm8k_per_example_{cond}.csv", index=False)
        gsm_rows[cond] = g
        per_mc[cond] = {t: mc_per_example(per[t]) for t in ("hellaswag", "boolq", "arc_easy")}
        for t, dfm in per_mc[cond].items():
            dfm.to_csv(TAB_DIR / f"{t}_per_example_{cond}.csv", index=False)
    if "baseline" not in agg:
        print("baseline not evaluated yet"); return
    cols = [("GSM8K flex EM", "gsm8k_cot/exact_match,flexible-extract"), ("GSM8K strict EM", "gsm8k_cot/exact_match,strict-match"),
            ("HellaSwag acc_norm", "hellaswag/acc_norm,none"), ("HellaSwag acc", "hellaswag/acc,none"),
            ("BoolQ acc", "boolq/acc,none"), ("ARC-E acc_norm", "arc_easy/acc_norm,none"), ("ARC-E acc", "arc_easy/acc,none")]
    rows = []
    for cond, res in agg.items():
        row = dict(condition=NICE[cond])
        for name, key in cols:
            v = res.get(key, np.nan) * 100
            row[name] = round(v, 2)
            row[f"Δ {name} (pp)"] = round(v - agg["baseline"].get(key, np.nan) * 100, 2)
        g = gsm_rows[cond]
        row["GSM8K truncated %"] = round(100 * g["truncated"].mean(), 2)
        row["GSM8K mean resp tokens"] = round(g["resp_tokens"].mean(), 1)
        rows.append(row)
    bench = pd.DataFrame(rows); bench.to_csv(TAB_DIR / "benchmarks.csv", index=False)
    print(bench.to_string(index=False))
    # ---------- controlled paired comparisons (final checkpoints) ----------
    comps = [("lora16", "lora4", "rank"), ("l2sp", "full_ft", "explicit regularisation"),
             ("oplora16", "lora16", "orthogonal protection"), ("lora16", "full_ft", "Full FT vs LoRA-16"),
             ("full_ft", "full_ft_mid", "Full FT progression")]
    crow = []
    for a, b, why in comps:
        if a not in gsm_rows or b not in gsm_rows:
            continue
        ea = gsm_rows[a]["em_flexible-extract"].values; eb = gsm_rows[b]["em_flexible-extract"].values
        crow.append(dict(comparison=f"{NICE[a]} − {NICE[b]}", factor=why, benchmark="GSM8K", **dict(zip(["diff_pp", "ci_lo", "ci_hi"], paired_boot(ea, eb)))))
        for t, metric in (("hellaswag", "acc_norm"), ("boolq", "acc"), ("arc_easy", "acc_norm")):
            crow.append(dict(comparison=f"{NICE[a]} − {NICE[b]}", factor=why, benchmark=t,
                             **dict(zip(["diff_pp", "ci_lo", "ci_hi"], paired_boot(per_mc[a][t][metric].values, per_mc[b][t][metric].values)))))
    pd.DataFrame(crow).round(2).to_csv(TAB_DIR / "paired_comparisons.csv", index=False)
    # GSM8K: separate extraction/truncation changes from correctness changes
    ext = []
    for cond, g in gsm_rows.items():
        ext.append(dict(condition=NICE[cond], flex_em=100 * g["em_flexible-extract"].mean(), strict_em=100 * g["em_strict-match"].mean(),
                        unextractable_flex_pct=100 * g["unextractable_flex"].mean(), truncated_pct=100 * g["truncated"].mean(),
                        em_among_not_truncated=100 * g.loc[~g["truncated"], "em_flexible-extract"].mean()))
    pd.DataFrame(ext).round(2).to_csv(TAB_DIR / "gsm8k_extraction.csv", index=False)
    # ---------- benchmark figure ----------
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
    for ax, (name, key) in zip(axes, [cols[0], cols[2], cols[4], cols[5]]):
        conds = [c for c in ALL_CONDITIONS if c in agg]
        vals = [agg[c].get(key, np.nan) * 100 for c in conds]
        ax.bar(range(len(conds)), vals, color=["#888"] + ["#4C72B0"] * (len(conds) - 1))
        ax.axhline(vals[0], color="k", lw=0.8, ls="--")
        ax.set_xticks(range(len(conds))); ax.set_xticklabels([NICE[c] for c in conds], rotation=45, ha="right", fontsize=8)
        ax.set_title(name); ax.set_ylabel("%")
    fig.tight_layout(); fig.savefig(FIG_DIR / "benchmarks.png", dpi=200); plt.close(fig)
    # ---------- learning curves ----------
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
    for m in METHODS:
        lg = read_jsonl(RUNS_DIR / m / "train_log.jsonl")
        if not lg:
            continue
        df = pd.DataFrame(lg)
        axes[0].plot(df["update"], df["sft_loss"].rolling(10, min_periods=1).mean(), label=NICE[m], lw=1)
        axes[1].plot(df["update"], df["lr"], label=NICE[m], lw=1)
        if m == "l2sp":
            axes[2].plot(df["update"], df["sft_loss"].rolling(10, min_periods=1).mean(), label="SFT loss", lw=1)
            axes[2].plot(df["update"], df["total_loss"].rolling(10, min_periods=1).mean(), label="total = SFT + λΩ", lw=1)
            ax2 = axes[2].twinx(); ax2.plot(df["update"], df["lam_omega"], color="C3", lw=1, label="λΩ"); ax2.set_ylabel("λΩ", color="C3")
    axes[0].set_title("SFT loss (10-update moving avg)"); axes[0].set_xlabel("optimizer update"); axes[0].legend(fontsize=8)
    axes[1].set_title("learning rate"); axes[1].set_xlabel("optimizer update"); axes[1].set_yscale("log")
    axes[2].set_title("L2-SP: SFT vs total objective, λΩ"); axes[2].set_xlabel("optimizer update"); axes[2].legend(fontsize=8, loc="upper center")
    fig.tight_layout(); fig.savefig(FIG_DIR / "learning_curves.png", dpi=200); plt.close(fig)
    # ---------- regularisation + resources ----------
    res_rows = []
    for m in METHODS:
        f = RUNS_DIR / m / "summary.json"
        if not f.exists():
            continue
        s = jload(f)
        lg = pd.DataFrame(read_jsonl(RUNS_DIR / m / "train_log.jsonl"))
        row = dict(method=NICE[m], trainable_params=s["n_trainable"], train_time_h=round(s["train_time_s"] / 3600, 3),
                   peak_alloc_gb=s["peak_alloc_gb"], peak_reserved_gb=s["peak_reserved_gb"],
                   recovery_ckpt_gb=max(s["recovery_ckpt_gb"].values()) if s["recovery_ckpt_gb"] else None,
                   skipped_updates=len(s["skipped_updates"]), final_sft_loss_avg_last50=round(lg["sft_loss"].tail(50).mean(), 4))
        e = s["export"]
        if METHODS[m]["kind"] == "full":
            row.update(export_fp16_gb=gb(e["export_bytes"]), displacement_final=e["displacement_l2"],
                       penalty_overhead_s=s["penalty_time_s"], theta0_gb=gb(s["overhead"].get("theta0_bytes", 0)))
            if m == "full_ft" and (RUNS_DIR / m / "export_mid.json").exists():
                row["displacement_mid"] = jload(RUNS_DIR / m / "export_mid.json")["displacement_l2"]
            if m == "l2sp":
                row.update(omega_final=lg["omega"].iloc[-1], lam_omega_final=lg["lam_omega"].iloc[-1])
        else:
            row.update(adapter_gb=gb(e["adapter_bytes"]), adapter_export_gb=gb(e["export_bytes"]), merged_fp16_gb=gb(e["merged_bytes"]),
                       merge_max_abs_logit_diff=e["merge_check"]["max_abs_logit_diff"], merge_top1_agree=e["merge_check"]["top1_agreement"])
            if METHODS[m]["kind"] == "oplora":
                row.update(svd_preprocess_s=s["overhead"].get("svd_preprocess_s"), basis_gb=gb(s["overhead"].get("basis_bytes", 0)),
                           export_factor_s=e["export_factor_s"], max_rel_UkT_deltaW=e["max_rel_UkT_deltaW"])
        res_rows.append(row)
    pd.DataFrame(res_rows).to_csv(TAB_DIR / "resources.csv", index=False)
    print(pd.DataFrame(res_rows).to_string(index=False))
    # ---------- diagnostics figure ----------
    if (TAB_DIR / "diagnostics.csv").exists():
        dg = pd.read_csv(TAB_DIR / "diagnostics.csv")
        fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
        for ax, col, ttl in zip(axes, ["rho_k", "eta", None], ["ρ_k (k=128)", "η = ‖ΔW‖/‖W0‖", "R(z): math vs non-math"]):
            for i, L in enumerate(DIAG_LAYERS):
                sub = dg[dg.layer == L]
                x = np.arange(len(sub)) + i * 0.27
                if col:
                    ax.bar(x, sub[col], width=0.27, label=f"layer {L}")
                else:
                    ax.scatter(x, sub["R_math_mean"], marker="o", label=f"L{L} math")
                    ax.scatter(x, sub["R_nonmath_mean"], marker="x", label=f"L{L} non-math")
            ax.set_xticks(np.arange(len(sub)) + 0.27); ax.set_xticklabels([NICE[c] for c in sub["condition"]], rotation=45, ha="right", fontsize=8)
            ax.set_title(ttl); ax.legend(fontsize=7)
            if col == "eta" or col is None:
                ax.set_yscale("log")
        fig.tight_layout(); fig.savefig(FIG_DIR / "diagnostics.png", dpi=200); plt.close(fig)
    # ---------- qualitative: all 168 responses + 42-response annotation sheet ----------
    allq = []
    for cond in ALL_CONDITIONS:
        f = EVAL_DIR / cond / "qualitative.json"
        if f.exists():
            allq += jload(f)
    if allq:
        q = pd.DataFrame(allq)
        def json_ok(s):
            try:
                json.loads(re.sub(r"^```(json)?|```$", "", s.strip()).strip()); return True
            except Exception:
                return False
        def distinct3(s):
            w = s.split(); g = list(zip(w, w[1:], w[2:]))
            return len(set(g)) / len(g) if g else 1.0
        q["json_valid"] = [json_ok(s) if f == "json" else None for s, f in zip(q.response, q.format)]
        q["template_keys_present"] = [bool(re.search(r"^\s*\w+\s*=", s, re.M)) if f == "template" else None for s, f in zip(q.response, q.format)]
        q["distinct_trigram_ratio"] = q.response.map(distinct3)
        q["math_markers"] = q.response.map(lambda s: len(re.findall(r"\\boxed|\\frac|=\s*\d|\$|The answer is|step \d", s, re.I)))
        q["condition_name"] = q.condition.map(NICE)
        q = q.sort_values(["question", "format", "condition"])
        q.to_csv(TAB_DIR / "qualitative_all_168.csv", index=False)
        plain_order = [it["question"] for it in load_qual_prompts() if it["format"] == "plain"]
        subset = q[q.question.isin([plain_order[0], plain_order[-1]])].copy()
        for c in ["format_compliance(0-2)", "coherence_relevance(0-2)", "repetition(y/n)", "unnecessary_math_framing(y/n)",
                  "inappropriate_content(y/n)", "excerpt", "notes"]:
            subset[c] = ""
        subset.to_csv(TAB_DIR / "qualitative_annotation_42.csv", index=False)
        print(f"qualitative: {len(q)} responses saved; annotation sheet has {len(subset)} rows "
              f"(questions: {plain_order[0]}, {plain_order[-1]})")
    print("tables →", TAB_DIR, "| figures →", FIG_DIR)

In [14]:
if RUN_ANALYSIS:
    run_analysis()

      condition  GSM8K flex EM  Δ GSM8K flex EM (pp)  GSM8K strict EM  Δ GSM8K strict EM (pp)  HellaSwag acc_norm  Δ HellaSwag acc_norm (pp)  HellaSwag acc  Δ HellaSwag acc (pp)  BoolQ acc  Δ BoolQ acc (pp)  ARC-E acc_norm  Δ ARC-E acc_norm (pp)  ARC-E acc  Δ ARC-E acc (pp)  GSM8K truncated %  GSM8K mean resp tokens
       Baseline          43.37                  0.00              0.0                     0.0               47.33                       0.00          37.57                  0.00      64.43              0.00           55.89                   0.00      60.82              0.00               0.83                   226.0
  Full FT (mid)          56.71                 13.34              0.0                     0.0               45.74                      -1.59          37.13                 -0.44      61.65             -2.78           62.54                   6.65      65.66              4.84               0.83                   142.5
        Full FT          64.75                